In [11]:
import os
import sqlite3
import argparse
from datasets import load_dataset

In [12]:
from tabulate import tabulate 

In [2]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /raid/phundh/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [10]:
import sys
import os
sys.path.insert(0, os.path.abspath('/raid/phundh/demo/spider'))
sys.path.insert(0, os.path.abspath('/raid/phundh/nltk_data'))

In [11]:
from spider.script_add_double_flashes import *

['t1', 't2']
['select', 'count', '(', '*', ')', 'from', 'người', 'tham', 'gia', 'as', 't1', 'join', 'người', 'tham', 'gia', 'sự', 'kiện', 'as', 't2', 'on', 't1.id', 'người', 'tham', 'gia', '=', 't2.id', 'người', 'tham', 'gia', 'where', 't1.chi', 'tiết', 'người', 'tham', 'gia', 'like', '"%Dr.%"']
select count ( * ) from "người tham gia" as t1 join "người tham gia sự kiện" as t2 on t1."id người tham gia" = t2."id người tham gia" where t1."chi tiết người tham gia" like "%Dr.%"


In [12]:
from evaluation import Evaluator
from process_sql import get_schema, Schema, get_sql
import sqlite3
import os

In [17]:
os.makedirs("db_demo", exist_ok=True)

sample = {
    "schema": """CREATE TABLE IF NOT EXISTS table_42529 (
        "Mùa giải" REAL,
        "Division" TEXT,
        "Thắng" REAL,
        "Thua" REAL,
        "Hòa" REAL,
        "Vị trí cuối cùng" TEXT,
        "Ghi chú" TEXT
    )
    """,
    "value":['INSERT INTO "table_42529" VALUES("2004","SPL","6","3","1","5","tồn tại")',
            'INSERT INTO "table_42529" VALUES("2004","SPL","5","4","1","6","tồn tại")',
            'INSERT INTO "table_42529" VALUES("2005","SPL","6","2","2","7","tồn tại")',
            'INSERT INTO "table_42529" VALUES("2002","SPL","6","2","2","7","tồn tại")'],
    "question": "",
    "query": """SELECT AVG ("Hòa") FROM table_42529 WHERE "Thắng" = '6' AND "Mùa giải" > '2004'""",
    "predict": """SELECT AVG ("Hòa") FROM table_42529 WHERE "Thắng" = '6' AND "Mùa giải" > '2004'""",
    "db_id": "example",
}

In [92]:
add_double_flashes(sample['schema'])

[]
['create', 'table', 'if', 'not', 'exists', 'table_42529', '(', '"Mùa giải"', 'real', ',', '"Division"', 'text', ',', '"Thắng"', 'real', ',', '"Thua"', 'real', ',', '"Hòa"', 'real', ',', '"Vị trí cuối cùng"', 'text', ',', '"Ghi chú"', 'text', ')']


'create table "if" not exists "table_42529" ( "Mùa giải" real" , "Division" text" , "Thắng" real" , "Thua" real" , "Hòa" real" , "Vị trí cuối cùng" text" , "Ghi chú" text" )'

In [22]:
def generate_db_file(db_path, schema: str, db_name: str = "example.sqlite",list_value=None):
    # Step 1: Create a directory named 'db' if it doesn't exist
    os.makedirs(db_path, exist_ok=True)

    # Step 2: Define the path for the SQLite database file
    db_path = os.path.join(db_path, db_name)

    # Step 3: Connect to the SQLite database (or create it if it doesn't exist)
    conn = sqlite3.connect(db_path)

    # Step 4: Create a cursor object
    cursor = conn.cursor()

    # Step 5: Execute the SQL command to create the table
    cursor.execute(schema)
    
    if list_value != None:
        for val in list_value:
            cursor.execute(val)
    
    # Step 6: Commit the changes
    conn.commit()

    # Step 7: Query the table names
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    # Step 8: Print the table names
    print(tables)

    # Step 9: Close the connection
    conn.close()

In [24]:
generate_db_file(
    db_path=f"db_demo/db/{sample['db_id']}",
    db_name=f"{sample['db_id']}.sqlite",
    schema=sample["schema"],
    list_value=sample["value"],
)

[('table_42529',)]


In [3]:
def execute_query(db,db_name, p_str):
    """
    return 1 if the values between prediction and gold are matching
    in the corresponding index. Currently not support multiple col_unit(pairs).
    """

    db_path = os.path.join(db, db_name)
    
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    try:
        res = cursor.execute(p_str)
        row = res.fetchall()
    # p_res = cursor.fetchall()
    # for row in res:
    #     print(row["mùa giải"])
        
    except:
        return False
    return row

In [10]:
result = execute_query(
    db=f"db_demo/db/transaction",
    db_name=f"transaction.sqlite",
    p_str="""SELECT * FROM 'sản phẩm'"""
)

for row in result:
    data_dict = {col: row[col] for col in row.keys()}
    print(data_dict)

{'mã sản phẩm': 1, 'tên sản phẩm': 'Bánh mì', 'giá': 15000.0, 'số lượng tồn': 50}
{'mã sản phẩm': 2, 'tên sản phẩm': 'Sữa tươi', 'giá': 20000.0, 'số lượng tồn': 30}
{'mã sản phẩm': 3, 'tên sản phẩm': 'Cà phê đen', 'giá': 25000.0, 'số lượng tồn': 40}
{'mã sản phẩm': 4, 'tên sản phẩm': 'Trà sữa', 'giá': 35000.0, 'số lượng tồn': 20}
{'mã sản phẩm': 5, 'tên sản phẩm': 'Nước cam', 'giá': 18000.0, 'số lượng tồn': 15}
{'mã sản phẩm': 6, 'tên sản phẩm': 'Kem vani', 'giá': 12000.0, 'số lượng tồn': 25}
{'mã sản phẩm': 7, 'tên sản phẩm': 'Bánh quy', 'giá': 10000.0, 'số lượng tồn': 60}
{'mã sản phẩm': 8, 'tên sản phẩm': 'Kẹo ngọt', 'giá': 5000.0, 'số lượng tồn': 100}
{'mã sản phẩm': 9, 'tên sản phẩm': 'Xúc xích', 'giá': 12000.0, 'số lượng tồn': 35}
{'mã sản phẩm': 10, 'tên sản phẩm': 'Bánh pizza', 'giá': 120000.0, 'số lượng tồn': 10}


In [16]:
conn = sqlite3.connect("db_demo/db/transaction/transaction.sqlite")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

res = cursor.execute("""SELECT * FROM 'giao dịch'""")
row = res.fetchall()

In [17]:
data = row

# Optional: Use tabulate for a well-formatted output
headers = [description[0] for description in cursor.description]
print(tabulate(data, headers=headers, tablefmt='pretty'))

# Close the connection

+--------------+----------------+-------------+--------------+-----------+
| mã giao dịch | ngày giao dịch | mã sản phẩm | số lượng mua | tổng tiền |
+--------------+----------------+-------------+--------------+-----------+
|      1       |   2024/10/20   |      1      |      2       |  30000.0  |
|      2       |   2024/10/20   |      3      |      1       |  25000.0  |
|      3       |   2024/10/21   |      2      |      3       |  60000.0  |
|      4       |   2024/10/21   |      5      |      2       |  36000.0  |
|      5       |   2024/10/21   |      8      |      5       |  25000.0  |
|      6       |   2024/10/22   |      6      |      3       |  36000.0  |
|      7       |   2024/10/22   |      7      |      4       |  40000.0  |
|      8       |   2024/10/22   |      4      |      1       |  35000.0  |
|      9       |   2024/10/23   |      9      |      2       |  24000.0  |
|      10      |   2024/10/23   |     10      |      1       | 120000.0  |
|      11      |   2024/1

In [9]:
result = execute_query(
    db=f"db_demo/db/transaction",
    db_name=f"transaction.sqlite",
    p_str="""SELECT * FROM 'giao dịch'"""
)

for row in result:
    data_dict = {col: row[col] for col in row.keys()}
    print(data_dict)

{'mã giao dịch': 1, 'ngày giao dịch': '2024/10/20', 'mã sản phẩm': 1, 'số lượng mua': 2, 'tổng tiền': 30000.0}
{'mã giao dịch': 2, 'ngày giao dịch': '2024/10/20', 'mã sản phẩm': 3, 'số lượng mua': 1, 'tổng tiền': 25000.0}
{'mã giao dịch': 3, 'ngày giao dịch': '2024/10/21', 'mã sản phẩm': 2, 'số lượng mua': 3, 'tổng tiền': 60000.0}
{'mã giao dịch': 4, 'ngày giao dịch': '2024/10/21', 'mã sản phẩm': 5, 'số lượng mua': 2, 'tổng tiền': 36000.0}
{'mã giao dịch': 5, 'ngày giao dịch': '2024/10/21', 'mã sản phẩm': 8, 'số lượng mua': 5, 'tổng tiền': 25000.0}
{'mã giao dịch': 6, 'ngày giao dịch': '2024/10/22', 'mã sản phẩm': 6, 'số lượng mua': 3, 'tổng tiền': 36000.0}
{'mã giao dịch': 7, 'ngày giao dịch': '2024/10/22', 'mã sản phẩm': 7, 'số lượng mua': 4, 'tổng tiền': 40000.0}
{'mã giao dịch': 8, 'ngày giao dịch': '2024/10/22', 'mã sản phẩm': 4, 'số lượng mua': 1, 'tổng tiền': 35000.0}
{'mã giao dịch': 9, 'ngày giao dịch': '2024/10/23', 'mã sản phẩm': 9, 'số lượng mua': 2, 'tổng tiền': 24000.0}
{

In [32]:
sample['db_id']

'example'

In [25]:
schema_dict = get_schema(
    os.path.join(f"db_demo/db/{sample['db_id']}", f"{sample['db_id']}.sqlite")
)
print("schema dict:", schema_dict)

schema dict: {'table_42529': ['mùa giải', 'division', 'thắng', 'thua', 'hòa', 'vị trí cuối cùng', 'ghi chú']}


In [14]:
import json

In [15]:
with open('db.json','r') as f:
    data = json.load(f)

In [16]:
data

{'name': 'battle_death',
 '"trận đánh"': {'word': ['CREATE TABLE"trận đánh"("nhận_dạng"int,"tên"text,"ngày"text,"chỉ_huy người bulgaria"text,"chỉ_huy tiếng latin"text,"kết_quả"text,primary key("nhận_dạng")) ; ',
   'INSERT INTO"trận đánh"VALUES(1,"trận chiến_adrianople","14 tháng 4 năm 1205","kaloyan","hói tôi",“ chiến_thắng của bulgaria ”) ; ',
   'INSERT INTO"trận đánh"VALUES(2,"trận chiến_serres","tháng 6 năm 1205","kaloyan","không xác_định",“ chiến_thắng của bulgaria ”) ; ',
   'INSERT INTO"trận đánh"VALUES(3,"trận chiến_rusion","31 tháng 1 năm 1206","kaloyan",“ thierry_de termond ”,“ chiến_thắng của bulgaria ”) ; ',
   'INSERT INTO"trận đánh"VALUES(4,"trận chiến_rodosto","tháng 2 năm 1206","kaloyan","không xác_định",“ chiến_thắng của bulgaria ”) ; ',
   'INSERT INTO"trận đánh"VALUES(5,"trận chiến_messinopolis","4 tháng 9 năm 1207","không xác_định","boniface của montferrat",“ chiến_thắng của bulgaria ”) ; ',
   'INSERT INTO"trận đánh"VALUES(6,"trận chiến_boruy","tháng 6 năm 1205","